# 01 · TA-LSTMAE: main results, ablation and uncertainty

Reproduces Tables 2–7 of the revised manuscript under the corrected, leakage-free pipeline (Section 4.8):
* **Main results** (seed 42): Table 4 row, raw confusion matrices, and Tables 5–6 temporal metrics. The **causal (onset-forward) MTTD is the primary metric**; the symmetric ±5-event value is reported only as a sensitivity result (Reviewer 1, comment 1).
* **Repeated independent runs** (5 seeds, mean ± SD and 95% CI) of every ablation configuration and of the causal-only model (R1-6, R1-5, R2-5).
* **MTTD sensitivity** to matching tolerance (0, ±1, ±3, ±5 events) and to the synthetic inter-arrival scale (R1-2).
* Checkpoints, per-run predictions, partition indices, timestamps, class weights, a threshold sweep, ECS alerts, an environment file and a SHA-256 manifest.

The controlled head-to-head comparison with re-implemented baselines is in `02_head_to_head_baselines.ipynb`. The measured end-to-end latency through Elasticsearch/Logstash is in `03_end_to_end_latency.ipynb`.

## 1. Setup, configuration and environment file

In [1]:
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
ROOT = Path.cwd() if (Path.cwd() / "talstmae").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import tensorflow as tf
from talstmae import Config, set_all_seeds
from talstmae import data as DT, models as M, metrics as MT, experiment as X, alerts as AL

SMOKE_TEST = False      # True = tiny synthetic data, ONLY to check that the code executes. Never report its numbers.
RESUME = True           # continue an interrupted run (finished models/seeds are skipped)
cfg = Config().smoke() if SMOKE_TEST else Config()
if os.environ.get("TALSTMAE_SETTINGS"):          # settings written by 00_RUN_ME.ipynb override the lines above
    _s = json.loads(Path(os.environ["TALSTMAE_SETTINGS"]).read_text())
    _base = Config(**{k: (tuple(v) if k in ("datasets", "seeds") else v) for k, v in _s.get("config", {}).items()})
    SMOKE_TEST, RESUME = bool(_s.get("smoke_test")), bool(_s.get("resume", True))
    cfg = _base.smoke() if SMOKE_TEST else _base
    print("settings from 00_RUN_ME:", _s)
if SMOKE_TEST:
    print("=" * 78 + "\nSMOKE TEST MODE: synthetic placeholder data. Results are INVALID and must not be reported.\n" + "=" * 78)
set_all_seeds(cfg.seed, cfg.deterministic_ops)
print("TensorFlow", tf.__version__, "| outputs ->", cfg.out.resolve())

cfg.save()
import sklearn, keras, platform
env = {"python": sys.version.split()[0], "platform": platform.platform(), "tensorflow": tf.__version__,
       "keras": keras.__version__, "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
       "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")]}
(cfg.out / "environment.json").write_text(json.dumps(env, indent=2))
(cfg.out / "environment.txt").write_text("\n".join(f"{k}=={env[k]}" for k in
                                         ("python", "tensorflow", "keras", "numpy", "pandas", "scikit-learn")) + "\n")
env

I0000 00:00:1794253291.371437   19840 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1794253292.303644   19840 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


settings from 00_RUN_ME: {'smoke_test': False, 'resume': True, 'latency_events': None, 'config': {'datasets': ['nsl_kdd', 'unsw_nb15', 'cicids2018'], 'seeds': [42, 43, 44]}}
TensorFlow 2.21.0 | outputs -> /tmp/claude-0/-home-claude/5b43f487-a08c-5b08-be88-1e2ff98cfe79/scratchpad/srv/ta-lstmae/outputs


{'python': '3.13.18',
 'platform': 'Linux-6.18.44-fc-v310-x86_64-with-glibc2.39',
 'tensorflow': '2.21.0',
 'keras': '3.15.1',
 'numpy': '2.5.3',
 'pandas': '3.0.5',
 'scikit-learn': '1.9.1',
 'gpu_devices': []}

## 2. Data: strict loading, preprocessing, contiguous splits, timestamps and leakage audit
Loaders never substitute placeholder data; a missing file raises an error. Expected model-input dimensions are 122 / 194 / 78. Note that NSL-KDD has **122** model-input features, so the compression ratio is 122→16 = 86.89% (Reviewer 1, comment 7). UNSW-NB15 has **9 attack categories plus Normal, i.e. 10 classes** (Reviewer 1, comment 8).

In [2]:
DATA = {ds: DT.prepare_dataset(ds, cfg) for ds in cfg.datasets}
WINDOW_COUNTS = {ds: DT.leakage_audit(DATA[ds], cfg) for ds in cfg.datasets}

[nsl_kdd]


  NSL-KDD train                  125,973  (expected 125,973)  OK
  NSL-KDD test                    22,544  (expected 22,544)  OK


  input_dim=122 OK | train=113,376 val=12,597 test=22,544 | class_weight={0: 0.9356, 1: 1.0739} | timestamps: synthetic exponential (mean=0.1s, seeds=1001/1501), resolution=Nones
[unsw_nb15]


  UNSW-NB15 train                175,341  (expected 175,341)  OK
  UNSW-NB15 test                  82,332  (expected 82,332)  OK
  note: UNSW-NB15 records permuted (seed 2024); label runs train 8,172 -> 76,562, test 5 -> 41,083


  input_dim=194 OK | train=157,807 val=17,534 test=82,332 | class_weight={0: 1.5642, 1: 0.7349} | timestamps: synthetic exponential (mean=0.1s, seeds=1002/1502), resolution=1.0s
[cicids2018]


  CICIDS-2018 attack categories: ['Benign', 'FTP-BruteForce', 'SSH-Bruteforce']
  CICIDS-2018 subset (1/6)       174,763  (expected 174,763)  OK


  input_dim=78 OK | train=110,102 val=12,233 test=52,428 | class_weight={0: 0.7375, 1: 1.5526} | timestamps: native flow timestamps (chronological split), resolution=1.0s


nsl_kdd     windows train/val/test = (113367, 12588, 22535)  OK | contiguous, disjoint, per-split windowing: OK | exact duplicate windows train∩test (first 20k): 0 | test label-run ratio 1.01 (random-like)


unsw_nb15   windows train/val/test = (157798, 17525, 82323)  OK | contiguous, disjoint, per-split windowing: OK | exact duplicate windows train∩test (first 20k): 0 | test label-run ratio 1.01 (random-like)


cicids2018  windows train/val/test = (110093, 12224, 52419)  OK | contiguous, disjoint, per-split windowing: OK | exact duplicate windows train∩test (first 20k): 0 | test label-run ratio 0.31 (chronological (genuine timestamps))


## 3. Model configurations and parameter counts (Table 2; Section 4.6)

In [3]:
display(M.parameter_check(cfg.seq_len))
TABLE2 = {}
for ds, D in DATA.items():
    ae, _ = M.build_tae(D["X_tr"].shape[1], cfg.latent_dim[ds])
    TABLE2[DT.NAMES[ds]] = M.param_table(ae).set_index("layer")["params"]
    with open(cfg.out / ds / "model_summary_TAE.txt", "w") as f:
        ae.summary(print_fn=lambda s, *a, **k: f.write(s + "\n"))
pd.DataFrame(TABLE2)

,configuration,input_dim,params,reported,check
0,TAE + CNN only,16,28833,28833.0,OK
1,TAE + Transformer only,16,81089,81089.0,OK
2,TAE + BiLSTM only,16,104001,104001.0,OK
3,TAE + LSTM (unidirectional),16,128033,128033.0,OK
4,TAE + HBLT (Full Proposed),16,179617,179617.0,OK
5,No Feature Selection + HBLT,122,261025,261025.0,OK
6,No Feature Selection + HBLT,78,227233,227233.0,OK
7,Causal-only (unidirectional LSTM),16,52417,NaN,—


,NSL-KDD,UNSW-NB15,CICIDS-2018
layer,,,
enc_dense256,31488,49920,20224
enc_bn,1024,1024,1024
enc_mha,1088,1088,1088
enc_layernorm,32,32,32
enc_bigru,3264,3264,3264
latent,528,759,528
dec_project,1088,1536,1088
dec_bigru,9600,9600,9600
dec_mha,8352,8352,8352


## 4. Main run (seed 42): TAE + HBLT per dataset
Checkpoints (`tae.keras`, `tae_encoder.keras`, `hblt_full.keras`), test predictions, confusion matrices and metrics are written to `outputs/<dataset>/`. These seed-42 checkpoints are reused by notebook 03.

In [4]:
RESULTS = {}
for ds, D in DATA.items():
    d = cfg.out / ds
    main_npz = d / "main_seed_predictions.npz"
    if RESUME and main_npz.exists() and (d / "hblt_full.keras").exists():
        _z = np.load(main_npz); p_val, p_te = _z["prob_val"], _z["prob_test"]
        print(f"{ds}: main run already done, loaded saved predictions (resume)")
    else:
        ae, enc, lat = X.train_tae(D, cfg, cfg.seed, save_dir=d)
        clf, h, p_val, p_te = X.train_window_classifier(D, lat, cfg, M.CONFIGS[M.FULL]["branches"], cfg.hblt_epochs[ds], cfg.seed)
        clf.save(d / "hblt_full.keras")
        with open(d / "model_summary_HBLT.txt", "w") as f:
            clf.summary(print_fn=lambda s, *a, **k: f.write(s + "\n"))
        pd.DataFrame(h.history).to_csv(d / "hblt_history.csv", index=False)
        np.savez_compressed(main_npz, prob_val=p_val, prob_test=p_te)
    y, ts = X.test_targets(D, cfg)
    res = MT.evaluate_predictions(y, p_te, ts, cfg.threshold, cfg.match_tolerance, cfg.t_response_s)
    RESULTS[ds] = dict(res=res, p_val=p_val, p_te=p_te)
    yhat = (p_te >= cfg.threshold).astype(np.int8)
    pd.DataFrame({"window_index": np.arange(len(y)), "record_index": np.arange(len(y)) + cfg.seq_len - 1,
                  "timestamp_s": ts, "y_true": y, "y_prob": p_te, "y_pred": yhat}).to_csv(d / "predictions_test.csv", index=False)
    (d / "confusion_matrix.json").write_text(json.dumps({k: res[k] for k in ("TN", "FP", "FN", "TP", "n")}, indent=2))
    X.save_json(res, d / "metrics.json")
    AL.write_bulk_ndjson((AL.ecs_document(ds, i, ts[i], p_te[i], res["MTTD_causal"], cfg.t_response_s)
                          for i in np.flatnonzero(yhat)), d / "alerts_ecs.ndjson")
    print(f"{ds:<11s} Acc={res['accuracy']:.4f} P={res['precision']:.4f} R={res['recall']:.4f} F1={res['f1']:.4f} | "
          f"TN={res['TN']:,} FP={res['FP']:,} FN={res['FN']:,} TP={res['TP']:,} | MTTD causal={res['MTTD_causal']:.4f}s "
          f"(symmetric {res['MTTD_symmetric']:.4f}s) APR={res['APR']:.4f} DR={res['DetectionRate_causal']:.4f}")

nsl_kdd: main run already done, loaded saved predictions (resume)


nsl_kdd     Acc=0.7846 P=0.9306 R=0.6718 F1=0.7803 | TN=9,064 FP=643 FN=4,210 TP=8,618 | MTTD causal=0.0661s (symmetric 0.0056s) APR=0.9306 DR=0.9751
unsw_nb15: main run already done, loaded saved predictions (resume)


unsw_nb15   Acc=0.8594 P=0.8439 R=0.9136 F1=0.8774 | TN=29,339 FP=7,657 FN=3,918 TP=41,409 | MTTD causal=0.0157s (symmetric 0.0005s) APR=0.8439 DR=0.9989
cicids2018: main run already done, loaded saved predictions (resume)


cicids2018  Acc=0.9992 P=0.9996 R=0.9986 F1=0.9991 | TN=30,217 FP=9 FN=31 TP=22,162 | MTTD causal=0.0000s (symmetric 0.0000s) APR=0.9996 DR=1.0000


## 5. Validation-set decision-threshold sweep (Section 4.8)
The threshold is chosen on validation windows only; the test set is never used for this choice.

In [5]:
rows = []
for ds, D in DATA.items():
    yv, _ = X.val_targets(D, cfg)
    for thr in np.round(np.arange(0.10, 0.91, 0.05), 2):
        m = MT.classification_metrics(yv, (RESULTS[ds]["p_val"] >= thr).astype(np.int8))
        rows.append(dict(dataset=ds, threshold=thr, **{k: m[k] for k in ("accuracy", "precision", "recall", "f1")}))
SWEEP = pd.DataFrame(rows); SWEEP.to_csv(cfg.out / "threshold_sweep.csv", index=False)
best = SWEEP.loc[SWEEP.groupby("dataset")["f1"].idxmax(), ["dataset", "threshold", "f1"]]
best.merge(SWEEP[SWEEP.threshold == 0.5][["dataset", "f1"]].rename(columns={"f1": "f1_at_0.5"}), on="dataset")

,dataset,threshold,f1,f1_at_0.5
0,cicids2018,0.10,0.997827,0.997477
1,nsl_kdd,0.55,0.986703,0.985614
2,unsw_nb15,0.15,0.955009,0.934967


## 6. Repeated independent runs: ablation and causal model with uncertainty (Table 7)
Every configuration is trained with each seed in `cfg.seeds`, with one TAE per (dataset, seed) shared across configurations. The table reports mean ± SD and the 95% t-interval. Paired tests across seeds compare each configuration with the full model, using Holm-adjusted p-values. Small differences whose intervals overlap should not be interpreted as architectural gains.

In [6]:
ABL = X.run_ablation(DATA, cfg, resume=RESUME)
ABL_SUM = MT.summarize_runs(ABL)
ABL_SUM.to_csv(cfg.out / "ablation_summary.csv", index=False)
display(MT.format_mean_sd(ABL_SUM, ("accuracy", "precision", "recall", "f1", "MTTD_causal")))
tests = [MT.paired_seed_test(ABL, "f1", c, M.FULL, ds) for ds in DATA for c in M.CONFIGS if c != M.FULL]
TESTS = pd.DataFrame(tests)
TESTS["paired_t_p_holm"] = np.nan
ok = TESTS.paired_t_p.notna()
TESTS.loc[ok, "paired_t_p_holm"] = MT.holm(TESTS.loc[ok, "paired_t_p"].values)
TESTS.to_csv(cfg.out / "ablation_paired_tests.csv", index=False); TESTS.round(4)

[2026-11-10 02:41:55] nsl_kdd seed=42: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] nsl_kdd seed=43: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] nsl_kdd seed=44: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] unsw_nb15 seed=42: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] unsw_nb15 seed=43: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] unsw_nb15 seed=44: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] cicids2018 seed=42: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] cicids2018 seed=43: all ablation configurations already done, skipped (resume)


[2026-11-10 02:41:55] cicids2018 seed=44: all ablation configurations already done, skipped (resume)


,dataset,model,runs,accuracy,precision,recall,f1,MTTD_causal
0,nsl_kdd,No Feature Selection + HBLT,3,0.7834 ± 0.0037,0.9305 ± 0.0110,0.6697 ± 0.0123,0.7787 ± 0.0056,0.0674 ± 0.0027
1,nsl_kdd,TAE + LSTM (unidirectional),3,0.7854 ± 0.0026,0.9288 ± 0.0100,0.6749 ± 0.0114,0.7817 ± 0.0046,0.0660 ± 0.0025
2,nsl_kdd,TAE + CNN only,3,0.7805 ± 0.0125,0.9418 ± 0.0231,0.6552 ± 0.0103,0.7727 ± 0.0120,0.0709 ± 0.0021
3,nsl_kdd,TAE + BiLSTM only,3,0.7566 ± 0.0054,0.9182 ± 0.0072,0.6285 ± 0.0111,0.7462 ± 0.0073,0.0768 ± 0.0027
4,nsl_kdd,TAE + Transformer only,3,0.5277 ± 0.0095,0.6788 ± 0.0111,0.3242 ± 0.0409,0.4376 ± 0.0351,0.1090 ± 0.0071
5,nsl_kdd,TAE + HBLT (Full Proposed),3,0.7806 ± 0.0028,0.9386 ± 0.0206,0.6582 ± 0.0182,0.7735 ± 0.0063,0.0698 ± 0.0045
6,nsl_kdd,Causal-only (unidirectional LSTM),3,0.7642 ± 0.0192,0.9285 ± 0.0305,0.6352 ± 0.0270,0.7540 ± 0.0218,0.0751 ± 0.0067
7,unsw_nb15,No Feature Selection + HBLT,3,0.8786 ± 0.0114,0.8625 ± 0.0263,0.9286 ± 0.0154,0.8940 ± 0.0072,0.0131 ± 0.0030
8,unsw_nb15,TAE + LSTM (unidirectional),3,0.8534 ± 0.0033,0.8239 ± 0.0060,0.9333 ± 0.0030,0.8752 ± 0.0021,0.0123 ± 0.0007
9,unsw_nb15,TAE + CNN only,3,0.8561 ± 0.0075,0.8317 ± 0.0167,0.9267 ± 0.0109,0.8765 ± 0.0044,0.0135 ± 0.0023


,dataset,metric,model_a,model_b,n_seeds,mean_diff,paired_t_p,wilcoxon_p,paired_t_p_holm
0,nsl_kdd,f1,No Feature Selection + HBLT,TAE + HBLT (Full Proposed),3,0.0053,0.1650,NaN,1.0000
1,nsl_kdd,f1,TAE + LSTM (unidirectional),TAE + HBLT (Full Proposed),3,0.0082,0.3229,NaN,1.0000
2,nsl_kdd,f1,TAE + CNN only,TAE + HBLT (Full Proposed),3,-0.0008,0.9479,NaN,1.0000
3,nsl_kdd,f1,TAE + BiLSTM only,TAE + HBLT (Full Proposed),3,-0.0273,0.0727,NaN,0.9454
4,nsl_kdd,f1,TAE + Transformer only,TAE + HBLT (Full Proposed),3,-0.3359,0.0035,NaN,0.0555
5,nsl_kdd,f1,Causal-only (unidirectional LSTM),TAE + HBLT (Full Proposed),3,-0.0195,0.3281,NaN,1.0000
6,unsw_nb15,f1,No Feature Selection + HBLT,TAE + HBLT (Full Proposed),3,0.0200,0.0480,NaN,0.7204
7,unsw_nb15,f1,TAE + LSTM (unidirectional),TAE + HBLT (Full Proposed),3,0.0012,0.3009,NaN,1.0000
8,unsw_nb15,f1,TAE + CNN only,TAE + HBLT (Full Proposed),3,0.0025,0.3685,NaN,1.0000
9,unsw_nb15,f1,TAE + BiLSTM only,TAE + HBLT (Full Proposed),3,-0.0038,0.5384,NaN,1.0000


## 7. MTTD sensitivity: matching tolerance and inter-arrival scale (Reviewer 1, comment 2)
The seed-42 predictions are held fixed. For NSL-KDD and UNSW-NB15 the test timestamps are regenerated with the same seed at several mean inter-arrival times. CICIDS-2018 uses its native timestamps only. **Causal** is the primary rule.

In [7]:
SENS = []
for ds, D in DATA.items():
    y, _ = X.test_targets(D, cfg)
    yhat = (RESULTS[ds]["p_te"] >= cfg.threshold).astype(np.int8)
    scales = [None] if D["native_timestamps"] else [0.01, 0.05, 0.1, 0.5, 1.0]
    for mean in scales:
        if mean is None:
            ts, label = D["ts_te"][cfg.seq_len - 1:], "native"
        else:
            raw = DT.synthetic_timestamps(len(D["y_te"]), cfg.ts_seeds.get(ds, 2000) + 500, mean)
            ts, label = DT.apply_resolution(raw, cfg.ts_resolution_s[ds])[cfg.seq_len - 1:], f"mean={mean}s"
        for tol in (0, 1, 3, 5):
            for causal in (True, False):
                t = MT.temporal_metrics(y, yhat, ts, tol=tol, causal=causal, t_response=cfg.t_response_s)
                SENS.append(dict(dataset=ds, timestamps=label, tolerance=tol, rule="causal" if causal else "symmetric", **t))
SENS = pd.DataFrame(SENS); SENS.to_csv(cfg.out / "mttd_sensitivity.csv", index=False)
SENS.pivot_table(index=["dataset", "timestamps", "rule"], columns="tolerance", values="MTTD").round(4)

tolerance                          0       1       3       5
dataset    timestamps rule                                  
cicids2018 native     causal     0.0  0.0000  0.0000  0.0000
                      symmetric  0.0  0.0000  0.0000  0.0000
nsl_kdd    mean=0.01s causal     0.0  0.0016  0.0047  0.0066
                      symmetric  0.0  0.0008  0.0010  0.0006
           mean=0.05s causal     0.0  0.0078  0.0234  0.0330
                      symmetric  0.0  0.0042  0.0051  0.0028
           mean=0.1s  causal     0.0  0.0157  0.0468  0.0661
                      symmetric  0.0  0.0084  0.0102  0.0056
           mean=0.5s  causal     0.0  0.0783  0.2338  0.3303
                      symmetric  0.0  0.0418  0.0508  0.0280
           mean=1.0s  causal     0.0  0.1566  0.4675  0.6606
                      symmetric  0.0  0.0836  0.1016  0.0561
unsw_nb15  mean=0.01s causal     0.0  0.0005  0.0012  0.0014
                      symmetric  0.0  0.0001  0.0001  0.0000
           mean=0.05s causal     0.0  0.0023  0.0061  0.0080
                      symmetric  0.0  0.0011  0.0008  0.0002
           mean=0.1s  causal     0.0  0.0050  0.0122  0.0157
                      symmetric  0.0  0.0024  0.0016  0.0005
           mean=0.5s  causal     0.0  0.0243  0.0620  0.0805
                      symmetric  0.0  0.0117  0.0070  0.0022
           mean=1.0s  causal     0.0  0.0472  0.1224  0.1600
                      symmetric  0.0  0.0231  0.0138  0.0046

## 8. Regenerated tables and comparison with the manuscript

In [8]:
TDIR = cfg.out / "tables"; TDIR.mkdir(exist_ok=True)
def save(df, name):
    df.to_csv(TDIR / f"{name}.csv", index=False)
    try: (TDIR / f"{name}.md").write_text(df.to_markdown(index=False))
    except ImportError: pass
    return df
save(pd.DataFrame(TABLE2).reset_index(), "table2_tae_parameters")
save(pd.DataFrame([dict(dataset=DT.NAMES[d], train_records=len(DATA[d]["y_tr"]) + len(DATA[d]["y_val"]),
                        test_records=len(DATA[d]["y_te"]), model_input_dim=DATA[d]["X_tr"].shape[1],
                        train_windows=WINDOW_COUNTS[d][0], val_windows=WINDOW_COUNTS[d][1], test_windows=WINDOW_COUNTS[d][2])
                   for d in DATA]), "table3_dataset_summary")
t4 = save(pd.DataFrame([dict(dataset=DT.NAMES[d], **{k: (round(v, 4) if isinstance(v, float) else v)
                        for k, v in RESULTS[d]["res"].items() if k in ("accuracy", "precision", "recall", "f1", "TN", "FP", "FN", "TP", "n")})
                        for d in DATA]), "table4_ta_lstmae_seed42")
full = ABL_SUM[ABL_SUM.model == M.FULL]
save(MT.format_mean_sd(full, ("accuracy", "precision", "recall", "f1")), "table4_ta_lstmae_mean_sd")
t6 = save(pd.DataFrame([dict(dataset=DT.NAMES[d], MTTD_causal_s=round(RESULTS[d]["res"]["MTTD_causal"], 4),
                             projected_MTTR_causal_s=round(RESULTS[d]["res"]["pMTTR_causal"], 4),
                             MTTD_symmetric_s_sensitivity=round(RESULTS[d]["res"]["MTTD_symmetric"], 4),
                             APR=round(RESULTS[d]["res"]["APR"], 4), DetectionRate=round(RESULTS[d]["res"]["DetectionRate_causal"], 4))
                        for d in DATA]), "table6_temporal_metrics_causal_primary")
save(ABL_SUM.round(4), "table7_ablation_mean_sd_ci")
REPORTED = {"nsl_kdd": dict(accuracy=0.7692, precision=0.9100, recall=0.6599, f1=0.7650, TN=8870, FP=837, FN=4363, TP=8465,
                            MTTD_causal=0.0667, MTTD_symmetric=0.0041, APR=0.9100),
            "unsw_nb15": dict(accuracy=0.8584, precision=0.8294, recall=0.9353, f1=0.8791, TN=28268, FP=8723, FN=2935, TP=42397,
                              MTTD_causal=0.0076, MTTD_symmetric=0.0000, APR=0.8294),
            "cicids2018": dict(accuracy=0.9994, precision=0.9985, recall=1.0000, f1=0.9993, TN=30127, FP=33, FN=0, TP=22259,
                               MTTD_causal=0.0000, MTTD_symmetric=0.0000, APR=0.9985)}
if not cfg.smoke_test:
    cmp = [dict(dataset=DT.NAMES[d], metric=k, reported=v, regenerated=round(float(RESULTS[d]["res"][k]), 4),
                difference=round(float(RESULTS[d]["res"][k]) - v, 4)) for d in DATA for k, v in REPORTED[d].items()]
    display(save(pd.DataFrame(cmp), "comparison_with_manuscript"))
display(t4); display(t6)

,dataset,metric,reported,regenerated,difference
0,NSL-KDD,accuracy,0.7692,0.7846,0.0154
1,NSL-KDD,precision,0.9100,0.9306,0.0206
2,NSL-KDD,recall,0.6599,0.6718,0.0119
3,NSL-KDD,f1,0.7650,0.7803,0.0153
4,NSL-KDD,TN,8870.0000,9064.0000,194.0000
5,NSL-KDD,FP,837.0000,643.0000,-194.0000
6,NSL-KDD,FN,4363.0000,4210.0000,-153.0000
7,NSL-KDD,TP,8465.0000,8618.0000,153.0000
8,NSL-KDD,MTTD_causal,0.0667,0.0661,-0.0006
9,NSL-KDD,MTTD_symmetric,0.0041,0.0056,0.0015


,dataset,accuracy,precision,recall,f1,TN,FP,FN,TP,n
0,NSL-KDD,0.7846,0.9306,0.6718,0.7803,9064,643,4210,8618,22535
1,UNSW-NB15,0.8594,0.8439,0.9136,0.8774,29339,7657,3918,41409,82323
2,CICIDS-2018,0.9992,0.9996,0.9986,0.9991,30217,9,31,22162,52419


,dataset,MTTD_causal_s,projected_MTTR_causal_s,MTTD_symmetric_s_sensitivity,APR,DetectionRate
0,NSL-KDD,0.0661,30.0661,0.0056,0.9306,0.9751
1,UNSW-NB15,0.0157,30.0157,0.0005,0.8439,0.9989
2,CICIDS-2018,0.0000,30.0000,0.0000,0.9996,1.0000


## 9. Output manifest (SHA-256 of inputs and outputs)

In [9]:
import datetime as dt
manifest = {"generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "smoke_test": cfg.smoke_test,
            "inputs": DT.PROVENANCE, "environment": env,
            "outputs": {str(p.relative_to(cfg.out)): DT.sha256(p) for p in sorted(cfg.out.rglob("*"))
                        if p.is_file() and p.name != "MANIFEST.json"}}
(cfg.out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
print(len(manifest["outputs"]), "files recorded in", cfg.out / "MANIFEST.json")

70 files recorded in outputs/MANIFEST.json
